# Network Analysis — Streckenveränderungen 2023–2025

Das Zürcher Tramnetz ist kein statisches Objekt.
Im Analysezeitraum 2023–2025 gab es mit dem Fahrplanwechsel Dezember 2023 den größten Netzausbau in der Geschichte der VBZ.
Dieses Notebook untersucht **was** sich verändert hat, **wo** und **wann** — und was das für die Pünktlichkeit bedeutet.

**Zentrale Fragen:**
1. Wo haben sich die meisten Änderungen abgespielt? — Haltestellen und Stadtteile
2. Wieviel hat sich verändert? — Quantifizierung pro Linie und gesamt
3. Wann fanden die Änderungen statt? — Zeitachse
4. Hat sich die Lage nach dem Ausbau verbessert oder verschlechtert? — Einlaufzeit neuer Abschnitte
5. Welche Knotenpunkte sind kritische Hotspots? — Kaskaden und Linienüberschneidungen
6. Welche Stadtteile profitieren? — Versorgungsqualität

**Rückschluss für alle weiteren Analysen:** → am Ende dieses Notebooks

## Setup

In [ ]:
from zh_tram_flow.notebook import *
import zh_tram_flow.analytics.network as an

TRAIN, TEST, lf, lf_all, lf_delay, lf_clean = setup_analysis("03_analysis_2-network")

# lf_all   — train + test features combined (all years)
# lf_delay — lf_all filtered: canceled == False
# lf_clean — analysis-ready: canceled=False · stop_sequence>1 · no Linie E/L50/L51
#             departure_delay / delay_delta masked to NaN for Nov 14–Dec 23 2025
#             (is_anomal flag added for transparency)
%load_ext autoreload
%autoreload 2

In [ ]:
log("Lade GTFS j23 / j24 / j25 ...")
gtfs, all_lines = an.load_gtfs(PATHS["root"])
success(f"{len(all_lines)} Linien geladen: {all_lines}")

In [ ]:
changes = an.build_changes_matrix(gtfs, all_lines)
success(f"Änderungsmatrix: {len(changes)} Linien")
show_df(changes[["line","n_j23","n_j24","n_j25","added_j24","removed_j24","added_j25","removed_j25"]].set_index("line"))

## Überblick — Das Netz im Wandel

Beim Fahrplanwechsel Dezember 2023 (j23 → j24) wurden **10 von 17 regulären Linien** im GTFS verändert — der größte Netzwechsel im VBZ-Analysezeitraum. j24 → j25 ist vergleichsweise stabil (nur L5 +5 neue Halte).

Die größten Änderungen betreffen L13 (+19 Halte), L11 (+13) und L9 (+8). Die Karte unten zeigt alle neuen (orange) und entfernten (grau) Haltestellen ab Dez 2023.

### Interaktive Karte

In [ ]:
an.plot_network_changes_map(changes)

**Beobachtung:** Die Karte zeigt die räumliche Konzentration der Änderungen: Orange-Cluster in der Innenstadt (Bahnhofstrasse-Achse, Paradeplatz) sowie an den neuen Endabschnitten — L13 Richtung Sihlcity im Süden, L11 Richtung Rehalp im Osten. Grau (entfernte Halte) taucht nur vereinzelt auf — der Wechsel war überwiegend additiv, keine grossen Streckenabschnitte wurden gestrichen.

## Wo? — Räumliche Verteilung der Änderungen

In [ ]:
section_header("Neue Haltestellen nach Stadtkreis")
an.plot_new_stops_by_district(changes, lf_all, cfg)
show_df(an.table_new_stops_by_district(changes, lf_all))

In [ ]:
log("Netto-Änderungen pro Linie (j23→j24 und j24→j25)")

In [ ]:
show_df(an.table_network_netto_changes(changes))

**Beobachtung:** Das Balkendiagramm zeigt, dass die meisten neuen GTFS-Haltestellen in den Innenstadt-Kreisen erscheinen — Kreis 1 mit 12 neuen Haltestellen führt. Das ist zunächst überraschend, lässt sich aber nach Prüfung der konkreten Haltestellen-Namen erklären.

**Top-Kreise nach neuen GTFS-Haltestellen (ab Dez 2023):**
| Stadtkreis | Neue Halte |
|:---|---:|
| Kreis 1 (Altstadt/City) | **12** |
| Kreis 3 (Wiedikon) | 10 |
| Kreis 6 (Unterstrass) | 10 |
| Kreis 7 (Fluntern/Witikon) | 9 |
| Kreis 2 (Enge/Wollishofen) | 6 |
| Kreis 8 (Riesbach) | 5 |

**Was steckt dahinter — zwei verschiedene Ursachen:**

1. **GTFS-Artefakt (K1, K6):** Die meisten "neuen" Halte in Kreis 1 sind Stammhaltestellen, die seit Jahrzehnten bestehen — Paradeplatz, Bahnhofstrasse/HB, Rennweg, Stockerstrasse, Tunnelstrasse. Sie erscheinen als "neu", weil L6, L8, L9, L13 ab j24 andere repräsentative Trip-Shapes im GTFS nutzen, die durch diese Innenstadt-Achse führen. **Keine neue Infrastruktur.**

2. **Echte Streckenerweiterungen (K3, K8):**
   - **L13 → Sihlcity/Süd (K3):** Albisgütli, Laubegg, Eschergutweg, Waidfussweg, Sihlcity Nord, Uetlihof — echte neue Endabschnitte im Süden
   - **L11 → Rehalp/Zürichberg (K8):** Balgrist, Rehalp, Burgwies, Friedhof Enzenbühl, Hedwigsteig, Hegibachplatz — Verlängerung Richtung Zürichberg

**Kritischer Kontrast:**  
Die tatsächlichen Problemkreise K11 (Schwamendingen) und K12 (Oerlikon) — beide Spitze bei Verspätung in der Spatial-Analyse — haben **keine** neuen Haltestellen erhalten. Der Netzausbau zielte in Richtungen, die ohnehin gut performen.

## Wieviel? — Quantifizierung der Änderungen

In [ ]:
an.plot_network_stop_count_by_line(changes, cfg)

In [ ]:
show_df(an.table_network_netto_changes(changes))

**Beobachtung:** Das Quantifizierungs-Chart und die Tabelle liefern klare Zahlen zum Ausmass der GTFS-Änderungen.

**Netto-Änderungen j23 → j24 (die grossen Bewegungen):**
| Linie | j23 | j24 | j25 | Δ j23→j24 | Anmerkung |
|:---|---:|---:|---:|---:|:---|
| **L13** | 11 | 30 | 30 | **+19** | Grösste Änderung (+173%) |
| **L11** | 20 | 33 | 34 | **+13** | +16 hinzu, -3 entfernt |
| **L9** | 24 | 32 | 32 | **+8** | +13 hinzu, -5 entfernt |
| L18 | 0 | 16 | 0 | +16 | Temporäre Linie — nur 2024 aktiv |
| **L6** | 24 | 16 | 16 | **−8** | Netto 8 Halte entfernt |

**Anomalie L2:** j23=31 Halte → j24=21 (−10) → j25=31 (+10 zurück). Die Haltestellen-Zahl schwankt stark zwischen den Jahren, obwohl L2 als "stabil" gilt. Wahrscheinlich eine GTFS-Routing-Variante (andere Wendeäste oder Kursführung je Fahrplan), keine reale Streckenkürzung und Rückkehr.

**Neu in j25 (nicht j24):** L5 wächst von 9 auf 14 Halte (+5) — der einzige nennenswerte Ausbau beim j24→j25-Übergang.

**Stabil über alle Jahre:** L10, L12, L14, L17 (identische Haltestellen-Zahl in j23/j24/j25) — echte Referenzlinien für jahresübergreifende Vergleiche.

## Wann? — Zeitachse der Änderungen

In [ ]:
an.plot_monthly_delay_all_lines(lf_all, cfg)

**Legende Vertikale Linien:**
| Linie | Bedeutung |
|:---|:---|
| Rot gestrichelt | Fahrplanwechsel Dez 2023 (j23 → j24) — größte Netzreorganisation im Analysezeitraum |
| Grau gepunktet | Jahreswechsel Jan 2024 / Jan 2025 — zur Orientierung |

In [ ]:
show_df(an.table_delay_before_after_switch(lf_all))

**Beobachtung:** Kein erkennbarer Knick oder Sprung bei Jan 2024 (Fahrplanwechsel). Die Zeitreihen für alle Linien verlaufen kontinuierlich über den Wechsel hinweg.

**Veränderte vs. stabile Linien — Δ vor/nach Fahrplanwechsel:**
| Linie | Typ | vor (2023) | nach (2024–25) | Δ |
|:---|:---|---:|---:|---:|
| L11 | ✦ verändert | 65.1s | 70.4s | **+5.3s** |
| L13 | ✦ verändert | 51.6s | 53.0s | +1.4s |
| L9 | ✦ verändert | 58.7s | 54.3s | **−4.4s** |
| L15 | stabil | 57.9s | 63.1s | **+5.2s** |
| L8 | stabil | 52.6s | 54.7s | +2.1s |
| L12 | stabil | 53.1s | 51.2s | −1.9s |

**Kernbefund:** Die veränderten Linien zeigen keine einheitliche Richtung — L11 stieg um +5.3s, L9 verbesserte sich um −4.4s. Noch deutlicher: L15 (stabil, keine GTFS-Änderung) erhöhte sich um +5.2s — praktisch identisch mit L11 (+5.3s). Da stabile und veränderte Linien die gleiche Bandbreite an Veränderungen zeigen, liegt die Quelle der Variation nicht im Netzwechsel selbst, sondern in externen Faktoren (saisonale Muster, Fahrgastzuwachs, Wetter).

**Starkes Finding:** Die VBZ hat den grössten Fahrplanwechsel in der Netzgeschichte ohne erkennbare Verspätungs-Disruption durchgeführt. Der Netzwechsel ist im Delay-Signal nicht sichtbar.

## Einlaufzeit — Performen neue Abschnitte anders?

In [ ]:
section_header("Einlaufzeit — neue vs. bestehende Haltestellen")
an.plot_einlaufzeit(changes, lf_all, cfg)

In [ ]:
log("Einlaufzeit Tabelle")
show_df(an.table_einlaufzeit(changes, lf_all))

**Beobachtung:** Die Einlaufzeit-Charts zeigen kein einheitliches Muster — das Ergebnis ist linienabhängig und erklärt sich aus der Lage der Haltestellen.

**Neue vs. bestehende Haltestellen — Ø Delay ab Jan 2024:**
| Linie | Bestehende Halte (s) | Neue Halte (s) | Δ neu−best. |
|:---|---:|---:|---:|
| L11 (verändert) | 75.5 | 68.2 | **−7.3s** (neue besser) |
| L9 (verändert) | 57.4 | 50.8 | **−6.6s** (neue besser) |
| L13 (verändert) | 52.1 | 53.3 | +1.2s (≈ gleich) |
| L10 | 57.6 | 66.4 | **+8.8s** (neue schlechter) |
| L8 | 55.5 | 63.4 | **+7.9s** (neue schlechter) |
| L7 | 62.9 | 54.3 | −8.6s (neue besser) |
| L2 | 56.9 | 53.7 | −3.2s (neue besser) |

**Warum performen die "neuen" L11/L9-Halte besser?**  
L11 und L9 haben im GTFS vor allem Innenstadthalte als "neu" — Paradeplatz, Bellevue, Bürkliplatz, Bahnhof Stadelhofen. Das sind gut ausgebaute, häufig angefahrene Knotenpunkte mit stabiler Infrastruktur. Dass sie besser performen als die "bestehenden" Halte (die eher die Endabschnitte im problematischen K11/K12 abdecken) bestätigt den Befund aus der Stadtkreis-Analyse: Das Randlagen-Problem existiert unabhängig vom Netzwechsel.

**Echte Erweiterungen (L13 Sihlcity, L11 Rehalp):** Für L13 ist Δ ≈ 0 (+1.2s) — die neuen Südabschnitte performen wie der Rest der Linie. Kein Einlauf-Effekt, aber auch kein Einbruch.

**Linien ohne GTFS-Änderung (L10, L8):** Δ +8.8s / +7.9s — die "neuen" GTFS-Shapes dieser Linien führen zufällig durch schlechtere Abschnitte. Kein kausaler Zusammenhang mit echten Streckenerweiterungen.

> **Fazit:** Kein klassischer Einlauf-Effekt nachweisbar. Die Unterschiede sind lagebezogen, nicht zeitbezogen.

## Hotspots & Kaskaden — Kritische Knotenpunkte

In [ ]:
section_header("Hotspots & Kaskaden")
an.plot_hotspots(changes, lf_all, cfg)

In [ ]:
log("Hotspot Tabelle")
show_df(an.table_hotspots(changes, lf_all))

**Beobachtung:** Die Knotenpunkte mit den meisten Linien (Central: 7, Paradeplatz: 7) haben beide einen Ø Delay von ca. 49s — das liegt **unter** dem Gesamtdurchschnitt (~55s).

**Top-Hotspots nach Linienanzahl (j25):**
| Haltestelle | Linien | Ø Delay (s) | Beobachtungen |
|:---|---:|---:|---:|
| Central | 7 | 49.1 | 1 081 588 |
| Paradeplatz | 7 | 49.0 | 1 270 955 |
| Stauffacher | 6 | 60.2 | 986 075 |
| Bahnhofplatz/HB | 5 | 46.3 | 474 041 |
| Bürkliplatz | 5 | 52.3 | 917 620 |
| Bahnhofquai/HB | 5 | 53.4 | 880 893 |
| Milchbuck | 5 | 61.3 | 747 434 |

**Kernbefund:** Es gibt **keine positive Korrelation** zwischen Linienanzahl und Verspätung — das Gegenteil scheint möglich. Die grossen Knotenpunkte (Central, Paradeplatz, Bahnhofplatz) liegen alle unter dem Durchschnitt. Das Kaskadenrisiko-Modell "mehr Linien = mehr Delay" findet in den Daten keine Bestätigung. Höhere Delays entstehen offenbar nicht an den zentralen Umsteigeknoten, sondern anderswo im Netz (→ weiterführend in `03_analysis_4-spatial`).

## Versorgungsqualität — Welche Stadtteile profitieren?

In [ ]:
section_header("Versorgungsqualität nach Stadtkreis")
an.plot_service_quality_by_district(lf_all, cfg)
show_df(an.table_service_quality_by_district(lf_all))

In [ ]:
an.plot_service_quality_district_map(lf_all)

**Beobachtung:** Der Chart zeigt die Veränderung der Linien-Anbindung pro Stadtkreis (Δ Anzahl verschiedener Linien, 2025 vs. 2023).

**Δ Linien pro Stadtkreis (j23 → j25):**
| Stadtkreis | j23 | j25 | Δ |
|:---|---:|---:|---:|
| Kreis 12 | 5 | 7 | **+2** |
| Kreis 4 | 11 | 13 | **+2** |
| Kreis 9 | 8 | 9 | +1 |
| Kreis 11 | 10 | 11 | +1 |
| Kreis 8 | 6 | 7 | +1 |
| Kreise 1, 2, 3, 5 | — | — | 0 |
| Kreis 10 | 4 | 3 | −1 |
| Kreis 6 | 14 | 13 | −1 |
| Kreis 7 | 11 | 9 | **−2** |

**Wichtiger Kontrast zu Stadtkreis-Chart oben:** Kreis 1 erhielt die meisten neuen Haltestellen (12), aber **null neue Linien** — die Innenstadt wird von denselben Linien bedient, die nun mehr Halte haben. Echte Anbindungs-Verbesserungen (neue Linien) liegen vor allem in Kreis 12 und Kreis 4.

**Verlierer:** Kreis 7 verliert 2 Linien (11→9). Kreis 6 und Kreis 10 verlieren je 1 Linie. Die Versorgungsqualität dieser Kreise hat sich im Betrachtungszeitraum verschlechtert.

## Liniencharakter-Profil — Strukturelle Kennzahlen im Vergleich

Jede Linie hat ein unverwechselbares strukturelles Profil. Fünf Dimensionen im Überblick:

| Dimension | Was sie zeigt |
|:---|:---|
| **Ø Halte / Fahrt** | Routenlänge — je länger, desto mehr Accumulation-Potenzial |
| **Stadtkreise (Anzahl)** | Geografische Reichweite — wie viele verschiedene Kreise werden durchfahren? |
| **Innenstadt-Anteil (%)** | Anteil Halte in Kreisen 1–5 — Exposition zur Innenstadtbelastung |
| **Strukturfaktor (s/Fahrt)** | Ø delay_delta × avg_stops — kumulierter Delay-Aufbau pro Fahrt |
| **Ø Arrival Delay (s)** | Sichtbares Ergebnis für Fahrgäste am Ende |

Heatmap: **Farbe normalisiert pro Spalte** — Dunkel = hoher Wert · Hell = niedriger Wert · Linienbeschriftung in VBZ-Farben.

In [ ]:
an.plot_line_profiles(lf_all)

## Fazit & Rückschluss auf die weitere Analyse

### Was die Netzanalyse für alle weiteren Notebooks bedeutet

Die GTFS-Analyse über 2023, 2024 und 2025 ergibt folgende strukturelle Erkenntnisse:

#### Verändertes Teilnetz — Jahresvergleich mit Vorbehalt
| Linie | j23 | j24 | j25 | Kontext |
| :---: | ---: | ---: | ---: | :--- |
| **9** | 24 Halte | 32 Halte | 32 Halte | +8 neue Abschnitte ab Dez 2023 |
| **11** | 20 Halte | 33 Halte | 34 Halte | +13 neue Abschnitte ab Dez 2023 |
| **13** | 11 Halte | 30 Halte | 30 Halte | +19 Halte (+173%) ab Dez 2023 |

Für diese Linien ist Linie 9 in 2023 strukturell eine **andere** Linie als Linie 9 in 2024–2025.

> **⚠️ Externe Recherche-Korrektur (Perplexity, Mai 2026):**
> Die VBZ-Medienmitteilung zum Fahrplanwechsel Dez 2023 nennt **keine formalen Streckenumbauten** für Tramlinien 9, 11, 13 — die ausgewiesenen Änderungen betrafen primär Buslinien. Die GTFS-Unterschiede könnten daher sein:
> - **Flexity-Rollout**: neue Fahrzeuge auf Linien 11/13 → präzisere Haltestellenaufzeichnung im GTFS
> - **Takt-/Umlaufänderungen**: mehr Kurse, andere Wendeäste → neue Stop-IDs im GTFS
> - **GTFS-Modellierungsartefakt**: Fahrplankopplungen die als neue Halte erscheinen
>
> Das bedeutet: Die "+173%" bei Linie 13 sind real im GTFS-Datensatz sichtbar, aber die Ursache ist unklar. Das `gtfs_year`-Feature ist trotzdem valide — es kodiert einen echten Zeitschnitt. Ob es Netzstruktur oder nur einen Zeiteffekt erfasst, zeigt der Modellvergleich.

#### Stabiles Referenznetz
Linien mit identischer Stoppanzahl j23 = j24 = j25: **L10, L12, L14, L17** — diese können jahresübergreifend direkt verglichen werden ohne Strukturbruch.

#### Das `gtfs_year`-Feature — kritische Bewertung

```python
# In 02_preparation.ipynb hinzufügen:
pl.when(pl.col("operating_date") < pl.lit("2024-01-01").str.to_date())
  .then(pl.lit("j23"))
  .otherwise(pl.lit("j24_j25"))
  .alias("gtfs_year")
```

**Was es codiert:** Den Zeitschnitt Dez 2023 — für Linien 9, 11, 13 gibt es im GTFS einen markanten Unterschied. Ob das eine echte Streckenänderung oder ein Modellierungsartefakt ist, ist offen.

**Limitierung:** Für strukturell stabile Linien ist `gtfs_year` eine reine Zeitvariable ohne Netz-Kontext. Ob es tatsächlich die Modellleistung verbessert, **muss empirisch getestet werden**.

**Alternative:** `n_stops_line` als kontinuierliches Signal — trifft den gleichen Sachverhalt, ohne die binäre Vereinfachung.

**Entscheidung:** Feature als Kandidat aufnehmen, im Modellvergleich evaluieren.

#### Hinweis für alle Analysis-Notebooks
> Alle Analysen in `03_analysis_4-spatial`, `03_analysis_3-temporal`, `03_analysis_5-meteo` und `03_analysis_6-events`
> sollten bei Linien-bezogenen Befunden den Netzwechsel Dezember 2023 als Kontextinformation nennen.
> Detaillierte Aufschlüsselung immer mit Verweis auf dieses Notebook: `03_analysis_2-network.ipynb`.

---

#### Offenes TODO: Kaskadenanalyse mit `trip_id` (F-NET-07)

Eine wichtige Folgefrage aus der Target-Analyse: **Wenn ein Trip mit Verspätung endet — startet der nächste Trip (selbes Fahrzeug, andere Richtung) dann ebenfalls zu spät?**

> **Wie die Analyse funktioniert:**
> VBZ plant an den Endpunkten Wendezeit ein (typisch 5–10 Min). Bei moderaten Verspätungen wird diese Wendezeit aufgebraucht und der nächste Trip startet pünktlich. Bei Extremverspätungen (> Wendezeit) kann die Verspätung auf den nächsten Trip übertragen werden — das ist der Kaskadeneffekt.
>
> Mit `trip_id` und der Sortierung nach `operating_date` + `stop_sequence` lässt sich für jeden Trip der letzte Stop-Delay extrahieren und mit dem ersten Stop-Delay des Nachfolge-Trips vergleichen.

**Warum das für die Modellierung wichtig ist:**
Ein Feature `prev_trip_end_delay` (Verspätung am Ende des vorherigen Trips) wäre ein starkes Vorhersage-Signal — insbesondere in den Abendstunden wenn sich Verspätungen aufschaukeln. Das könnte den 21h-Peak aus F-TEMP-01 teilweise erklären.

```python
# Skizze für 02_preparation oder Modellierungsphase:
# trip_end_delay = lf.group_by("trip_id").agg(
#     pl.col("arrival_delay").last().alias("trip_end_delay")
# )
# → join mit nächstem Trip über Fahrzeug-ID / Umlauf-ID
```

**Status:** Offen — Daten sind vorhanden (`trip_id` im Master-Set), Implementierung ausstehend. → F-NET-07

## Key Findings

→ Vollständige Findings-Tabelle mit Impact und Action in [`03_analysis_0-overview.ipynb`](03_analysis_0-overview.ipynb).

| ID | Finding | Präsentation | Status |
|:---|:---|:---|:---|
| F-NET-01 | Im GTFS zeigen L9/L11/L13 zum Fahrplanwechsel Dez 2023 markante Haltestellen-Zunahmen (+8/+13/+19). Keine formalen Tramstrecken-Umbauten dokumentiert — teils GTFS-Artefakte (neue Trip-Shapes durch Innenstadtachse), teils echte Erweiterungen (L13→Sihlcity, L11→Rehalp). | `story` | done |
| F-NET-02 | Stabile Referenzlinien (L10, L12, L14, L17): identische Haltestellen-Zahl über alle drei Jahre — direkte Jahresvergleiche möglich. Anomalie L2: 31→21→31 Halte (j24-Dip), wahrscheinlich GTFS-Routing-Variante. | `—` | done |
| F-NET-03 | `gtfs_year` Feature (`j23` vs `j24_j25`) kodiert den Zeitschnitt Dez 2023. Ob es Netzstruktur oder Zeiteffekt misst, zeigt der Modellvergleich. | `—` | done |
| F-NET-04 | Kein Einlaufzeit-Effekt nachweisbar: Unterschiede zwischen neuen und bestehenden GTFS-Haltestellen sind lagebedingt, nicht zeitbedingt. L11/L9 "neue" Halte = Innenstadtknoten → besser. L10/L8 "neue" Halte = problembelastete Abschnitte → schlechter. | `—` | done |
| F-NET-05 | Keine positive Korrelation zwischen Linienanzahl und Delay: Knotenpunkte Central und Paradeplatz (je 7 Linien) liegen bei 49s — unter dem Netz-Durchschnitt (~55s). Das Kaskadenrisiko-Modell findet in den Daten keine Bestätigung. | `hot` | done |
| F-NET-06 | Versorgungsqualität (Δ Linien): Kreis 12 (+2) und Kreis 4 (+2) gewinnen am meisten. Kreis 7 verliert 2 Linien. Kreis 1 erhielt die meisten neuen GTFS-Haltestellen (12), aber keine neuen Linien. | `—` | done |
| F-NET-07 | `trip_id` ermöglicht Kaskadenanalyse: Verspätungsübertragung von Fahrt zu Fahrt messbar — als Feature `prev_trip_end_delay` für die Modellierungsphase prüfen. | `—` | done |
| F-NET-08 | **Linie E** ist eine Entlastungs-/Verstärkerlinie mit 128.1s Ø Delay — klarer Ausreisser. Kein Datenfehler; im Modell behalten, aber als Sonderlinie annotieren. | `—` | done |
| F-NET-09 | **Netzausbau vs. Delay-Hotspots — kein Overlap:** Die echten Streckenerweiterungen (L13→Sihlcity K3, L11→Rehalp K8) und GTFS-Reorganisation (K1 Innenstadtachse) fanden in Kreisen statt, die ohnehin gut performen. Die Problemkreise K11 (Schwamendingen) und K12 (Oerlikon) erhielten keine neuen Haltestellen. Das Netz wurde ausgebaut, aber nicht dort wo es am meisten gebraucht würde. | `hot` | done |

## Export

In [ ]:
from pathlib import Path

img_dir = Path("../reports/img")
img_dir.mkdir(parents=True, exist_ok=True)

an.plot_new_stops_by_district(changes, lf_all, cfg, save_as=img_dir / "network-new-stops-by-district.png")
an.plot_network_stop_count_by_line(changes, cfg, save_as=img_dir / "network-stop-count-by-line.png")
an.plot_monthly_delay_all_lines(lf_all, cfg, save_as=img_dir / "network-monthly-delay-all-lines.png")
an.plot_einlaufzeit(changes, lf_all, cfg, save_as=img_dir / "network-einlaufzeit.png")
an.plot_hotspots(changes, lf_all, cfg, save_as=img_dir / "network-hotspots.png")
an.plot_service_quality_by_district(lf_all, cfg, save_as=img_dir / "network-service-quality-by-district.png")
an.plot_line_profiles(lf_all, save_as=img_dir / "network-line-profiles.png")
print("✅ Network exports saved to reports/img/")